In [ ]:
"""Day_21_Fusion_Model.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1P3RLpsoDeL177EGNto41mcopzdvQJYma

# Load Everything
"""

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BASE = '/content/drive/MyDrive/rare_disease_project/data'

In [ ]:
import pickle, json, os
import numpy as np
import pandas as pd
from ast import literal_eval
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (f1_score, accuracy_score,
                              matthews_corrcoef)
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import timm

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

In [ ]:
# ── Load symptom model components ──
with open(f'{BASE}/backend_models/tfidf_vectorizer.pkl', 'rb') as f:
    tfidf = pickle.load(f)
with open(f'{BASE}/backend_models/lr_symptoms_model.pkl', 'rb') as f:
    lr_model = pickle.load(f)
with open(f'{BASE}/backend_models/label_encoder_symptoms.pkl', 'rb') as f:
    le_sym = pickle.load(f)

In [ ]:
# ── Load image model ──
class ImageClassifier(nn.Module):
    def __init__(self, num_classes, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b4', pretrained=False,
            num_classes=0, global_pool='avg')
        feat_dim = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 512), nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.backbone(x))

In [ ]:
checkpoint = torch.load(
    f'{BASE}/image_model_best.pt',
    map_location=device,
    weights_only=False
)
le_img      = checkpoint['label_encoder']
NUM_CLASSES = checkpoint['num_classes']

In [ ]:
img_model = ImageClassifier(NUM_CLASSES).to(device)
img_model.load_state_dict(checkpoint['model_state'])
img_model.eval()

In [ ]:
print(f"✅ Symptom model loaded — {len(le_sym.classes_)} classes")
print(f"✅ Image model loaded   — {NUM_CLASSES} classes")
print(f"✅ Disease overlap      — 62/62 ✅")

In [ ]:
"""#   Load Multimodal Dataset"""

In [ ]:
import random

In [ ]:
# Load full dataset
df = pd.read_pickle(f'{BASE}/clean_multimodal_full.pkl')
df['orpha_code'] = df['orpha_code'].astype(str)

In [ ]:
if isinstance(df['symptoms'].iloc[0], str):
    df['symptoms'] = df['symptoms'].apply(literal_eval)

In [ ]:
df['symptom_text'] = df['symptoms'].apply(
    lambda s: ' [SEP] '.join(
        [x.strip().lower() for x in s if x.strip()]))

In [ ]:
# Filter to only the 62 common diseases
common_diseases = list(
    set(le_sym.classes_) & set(le_img.classes_))
df = df[df['orpha_code'].isin(common_diseases)].copy()

In [ ]:
# Verify images exist
images_base = f'{BASE}/zebramap/images'
def get_valid_image(row):
    images = row['images']
    if isinstance(images, str):
        images = literal_eval(images)
    for img in images:
        if isinstance(img, str):
            img = literal_eval(img)
        path = img.get('path', '')
        # Fix path if needed
        if '/zebramap/images/' in path:
            rel   = path.split('/zebramap/images/')[-1]
            path  = f"{images_base}/{rel}"
        if os.path.exists(path):
            return path
    return None

In [ ]:
print("Finding valid image paths...")
df['valid_image'] = df.apply(get_valid_image, axis=1)
df = df[df['valid_image'].notna()].copy()

In [ ]:
print(f"Multimodal samples (both modalities): {len(df):,}")
print(f"Unique diseases                      : {df['orpha_code'].nunique()}")

In [ ]:
# Sample max 50 per disease for speed
random.seed(42)
sampled = df.groupby('orpha_code').apply(
    lambda x: x.sample(n=min(50, len(x)), random_state=42),
    include_groups=False
).reset_index(level=0).reset_index(drop=True)

In [ ]:
print(f"Sampled dataset : {len(sampled):,} cases")
print(f"Avg per disease : {len(sampled)/sampled['orpha_code'].nunique():.1f}")

In [ ]:
"""# Train/Test Split"""

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# Encode labels
le_fusion = LabelEncoder()
le_fusion.fit(sampled['orpha_code'])
sampled['label'] = le_fusion.transform(sampled['orpha_code'])
NUM_FUSION_CLASSES = len(le_fusion.classes_)

In [ ]:
train_df, test_df = train_test_split(
    sampled,
    test_size    = 0.2,
    random_state = 42,
    stratify     = sampled['orpha_code']
)

In [ ]:
print(f"Fusion classes : {NUM_FUSION_CLASSES}")
print(f"Train samples  : {len(train_df):,}")
print(f"Test samples   : {len(test_df):,}")
print(f"Train diseases : {train_df['orpha_code'].nunique()}")
print(f"Test diseases  : {test_df['orpha_code'].nunique()}")

In [ ]:
"""#  Get Predictions From Both Models"""

In [ ]:
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225])
])

In [ ]:
def get_symptom_proba(df_subset):
    """Get symptom model probabilities for all rows"""
    vec   = tfidf.transform(df_subset['symptom_text'])
    proba = lr_model.predict_proba(vec)
    # Reorder to match le_fusion order
    sym_classes    = list(le_sym.classes_)
    fusion_classes = list(le_fusion.classes_)
    reorder_idx    = [sym_classes.index(c)
                      if c in sym_classes else 0
                      for c in fusion_classes]
    return proba[:, reorder_idx]

In [ ]:
def get_image_proba(df_subset):
    """Get image model probabilities for all rows"""
    img_model.eval()
    all_probs = []

    for _, row in df_subset.iterrows():
        try:
            img    = Image.open(row['valid_image']).convert('RGB')
            tensor = test_transforms(img).unsqueeze(0).to(device)
            with torch.no_grad():
                out  = img_model(tensor)
                prob = torch.softmax(out, dim=1)[0].cpu().numpy()
            # Reorder to match le_fusion order
            img_classes    = list(le_img.classes_)
            fusion_classes = list(le_fusion.classes_)
            reorder_idx    = [img_classes.index(c)
                              if c in img_classes else 0
                              for c in fusion_classes]
            all_probs.append(prob[reorder_idx])
        except Exception as e:
            all_probs.append(
                np.ones(NUM_FUSION_CLASSES) / NUM_FUSION_CLASSES)

    return np.array(all_probs)

In [ ]:
print("Getting symptom probabilities for test set...")
sym_proba_test = get_symptom_proba(test_df)
print(f"✅ Symptom proba shape: {sym_proba_test.shape}")

In [ ]:
print("Getting image probabilities for test set...")
img_proba_test = get_image_proba(test_df)
print(f"✅ Image proba shape  : {img_proba_test.shape}")

In [ ]:
y_test = test_df['label'].values
print(f"✅ Test labels shape  : {y_test.shape}")

In [ ]:
"""#  Late Fusion (Weighted Average)"""

In [ ]:
def evaluate_fusion(sym_proba, img_proba, labels,
                    w_sym=0.5, w_img=0.5):
    """Weighted average fusion"""
    fused   = w_sym * sym_proba + w_img * img_proba
    preds   = np.argmax(fused, axis=1)
    acc     = accuracy_score(labels, preds)
    f1      = f1_score(labels, preds,
                       average='macro', zero_division=0)
    top3    = sum(
        labels[i] in np.argsort(fused[i])[-3:]
        for i in range(len(labels))
    ) / len(labels)
    top5    = sum(
        labels[i] in np.argsort(fused[i])[-5:]
        for i in range(len(labels))
    ) / len(labels)
    mcc     = matthews_corrcoef(labels, preds)
    return acc, f1, top3, top5, mcc

In [ ]:
# ── Grid search for best weights ──
print("Grid searching best fusion weights...")
print(f"{'w_sym':6} {'w_img':6} {'Acc':8} {'F1':8} {'Top3':8}")
print("-" * 45)

In [ ]:
best_f1     = 0
best_weights = (0.5, 0.5)
results_grid = []

In [ ]:
for w_sym in np.arange(0.1, 1.0, 0.1):
    w_img = round(1.0 - w_sym, 1)
    acc, f1, top3, top5, mcc = evaluate_fusion(
        sym_proba_test, img_proba_test,
        y_test, w_sym, w_img
    )
    results_grid.append((w_sym, w_img, acc, f1, top3))
    print(f"{w_sym:6.1f} {w_img:6.1f} "
          f"{acc*100:7.2f}% {f1:8.4f} {top3*100:7.2f}%")
    if f1 > best_f1:
        best_f1      = f1
        best_weights = (w_sym, w_img)

In [ ]:
print(f"\n✅ Best weights: sym={best_weights[0]:.1f}, "
      f"img={best_weights[1]:.1f}")
print(f"✅ Best F1     : {best_f1:.4f}")

In [ ]:
"""# Final Fusion Results"""

In [ ]:
w_sym, w_img = best_weights
acc, f1, top3, top5, mcc = evaluate_fusion(
    sym_proba_test, img_proba_test,
    y_test, w_sym, w_img
)

In [ ]:
print("=" * 55)
print("FUSION MODEL — FINAL RESULTS")
print("=" * 55)
print(f"Fusion weights : sym={w_sym:.1f}, img={w_img:.1f}")
print(f"Accuracy       : {acc:.4f}  ({acc*100:.2f}%)")
print(f"Macro F1       : {f1:.4f}")
print(f"Top-3 Acc      : {top3:.4f}  ({top3*100:.2f}%)")
print(f"Top-5 Acc      : {top5:.4f}  ({top5*100:.2f}%)")
print(f"MCC            : {mcc:.4f}")
print(f"Test samples   : {len(y_test):,}")
print(f"Classes        : {NUM_FUSION_CLASSES}")
print("=" * 55)

In [ ]:
# ── Full comparison table ──
print("\n📊 ALL EXPERIMENTS COMPARISON")
print("=" * 65)
print(f"{'Model':30} {'Acc':8} {'F1':8} {'Top-3':8}")
print("-" * 65)
print(f"{'Exp1 Symptoms (5% data)':30} {'24.83%':8} {'0.2240':8} {'40.27%':8}")
print(f"{'Exp3 Symptoms (100% data)':30} {'34.73%':8} {'0.3463':8} {'54.76%':8}")
print(f"{'Exp3 Image (50/disease)':30} {'12.10%':8} {'0.1098':8} {'25.00%':8}")
print(f"{'Fusion (sym+img)':30} {f'{acc*100:.2f}%':8} {f'{f1:.4f}':8} {f'{top3*100:.2f}%':8}")
print("=" * 65)

In [ ]:
# Save results
fusion_results = {
    'experiment'   : 'fusion_late_weighted',
    'w_sym'        : w_sym,
    'w_img'        : w_img,
    'accuracy'     : acc,
    'macro_f1'     : f1,
    'top3_acc'     : top3,
    'top5_acc'     : top5,
    'mcc'          : mcc,
    'num_classes'  : NUM_FUSION_CLASSES,
    'test_samples' : len(y_test),
}
with open(f'{BASE}/fusion_results.pkl', 'wb') as f:
    pickle.dump(fusion_results, f)

In [ ]:
print("\n✅ Fusion results saved to Drive")

In [ ]:
"""# Real Patient Test"""

In [ ]:
def predict_fusion(symptoms_text, image_path,
                   w_sym=w_sym, w_img=w_img, top_k=5):
    """
    Full fusion prediction for one patient
    symptoms_text : comma separated symptoms string
    image_path    : path to medical image (or None)
    """
    # Symptom prediction
    text    = ' [SEP] '.join(
        [s.strip().lower() for s in
         symptoms_text.replace(',','\n').split('\n')
         if s.strip()])
    vec     = tfidf.transform([text])
    s_proba = lr_model.predict_proba(vec)[0]

    sym_classes    = list(le_sym.classes_)
    fusion_classes = list(le_fusion.classes_)
    reorder_idx    = [sym_classes.index(c)
                      if c in sym_classes else 0
                      for c in fusion_classes]
    s_proba = s_proba[reorder_idx]

    # Image prediction
    if image_path and os.path.exists(image_path):
        img    = Image.open(image_path).convert('RGB')
        tensor = test_transforms(img).unsqueeze(0).to(device)
        with torch.no_grad():
            out    = img_model(tensor)
            i_prob = torch.softmax(out, dim=1)[0].cpu().numpy()
        img_classes = list(le_img.classes_)
        r_idx       = [img_classes.index(c)
                       if c in img_classes else 0
                       for c in fusion_classes]
        i_proba = i_prob[r_idx]
        fused   = w_sym * s_proba + w_img * i_proba
        mode    = "multimodal"
    else:
        fused = s_proba
        mode  = "symptoms_only"

    top_idx = np.argsort(fused)[-top_k:][::-1]

    print(f"Mode: {mode}")
    print(f"Input: {symptoms_text}")
    print(f"\nTop {top_k} predictions:")
    print("-" * 50)
    for rank, idx in enumerate(top_idx, 1):
        orpha = le_fusion.inverse_transform([idx])[0]
        prob  = fused[idx] * 100
        conf  = "High" if fused[idx]>=0.5 \
                else "Medium" if fused[idx]>=0.2 \
                else "Low"
        print(f"  {rank}. ORPHA:{orpha:8} "
              f"{prob:6.1f}%  [{conf}]")

In [ ]:
# Test 1 — Retinitis Pigmentosa
print("=" * 55)
print("TEST 1 — Retinitis Pigmentosa")
predict_fusion(
    "night blindness, progressive visual field loss, "
    "bone spicule pigmentation, photophobia",
    None
)

In [ ]:
# Test 2 — Sarcoidosis
print("\n" + "=" * 55)
print("TEST 2 — Sarcoidosis")
predict_fusion(
    "bilateral hilar lymphadenopathy, dry cough, "
    "uveitis, fatigue, skin lesions",
    None
)

In [ ]:
# Test 3 — Marfan
print("\n" + "=" * 55)
print("TEST 3 — Marfan Syndrome")
predict_fusion(
    "tall stature, arachnodactyly, lens dislocation, "
    "aortic dilatation, pectus excavatum",
    None
)